# Week 4 Day 2 - LangGraph、オーケストレーションレイヤー

昨日は構成要素（building blocks）に触れました。今日はレイヤー2に進み、LangGraphがそれらの構成要素をグラフとして組み立て、状態（state）とメモリを管理しながら実行してくれる様子を見ていきます。

## LangGraphとは何か

LangGraphを使うと、一連の作業をグラフとして記述できます。ノードはただのPython関数であり、それらをエッジでつなぎ、次に何を実行するかを決めます。フレームワークは「state」と呼ばれる共有オブジェクトを保持し、それをノードからノードへと受け渡し、各ステップごとにスナップショットを保存できるので、アプリケーションは記憶し、回復することができます。

LangGraphがオーケストレーションする対象を選ばないということは、覚えておく価値があります。LLMが登場する必要すらないのです！

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">実行する前に</h2>
            <span style="color:#ff7800;"><code>OPENAI_API_KEY</code>に加えて、このラボではグラフに2つのツールを与えます。検索ツールには<a href="https://serper.dev">serper.dev</a>から取得する無料の<code>SERPER_API_KEY</code>が必要で、プッシュ通知ツールには、以前の週で設定した<code>PUSHOVER_TOKEN</code>と<code>PUSHOVER_USER</code>が必要です。これらすべてが<code>.env</code>ファイルに設定されていることを確認してください。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# いつも通り、まずはインポートと環境設定から

import os
import random
import requests
from typing import Annotated
from typing_extensions import TypedDict
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver

load_dotenv(override=True)

## State、そしてreducerについて

stateとは、グラフの中を流れていくオブジェクトです。これは`TypedDict`で記述します。各フィールドはそれぞれ独立して更新され、デフォルトでは新しい値が単純に古い値を置き換えます。

しかし、更新を上書きせずに積み重ねたい場合もあります。reducerとは、古い値と新しい値をどのように組み合わせるかを指定する小さな関数です。これは`Annotated`を使って付加します。LangGraphには、チャット向けのreducerとして`add_messages`があらかじめ用意されており、これは新しいメッセージを上書きするのではなく、進行中のリストに追加していきます。

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

## LLMを使わずに、5つのステップでグラフを作る

グラフの構築は常に同じ形をたどります。stateを定義し、builderを開始し、ノードを追加し、エッジを追加し、そしてcompileします。ここでは、仕組みそのものに集中できるように、ただ馬鹿げた一文を作るだけのノードを使って、全体を示します。

In [ ]:
nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Eels", "Pickles"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "sparkly", "haunted"]

def silly_node(state: State) -> dict:
    sentence = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    return {"messages": [{"role": "assistant", "content": sentence}]}

builder = StateGraph(State)
builder.add_node("silly", silly_node)
builder.add_edge(START, "silly")
builder.add_edge("silly", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "say something"}]})
print(result["messages"][-1].content)

これで全体のパターンが分かりました。グラフとは、実は単にPython関数をつなぎ合わせたものにすぎないということです。それでは、この馬鹿げたノードを、モデルを呼び出すノードに置き換えてみましょう。

In [ ]:
llm = ChatOpenAI(model="gpt-5.4-mini")

def chatbot_node(state: State) -> dict:
    return {"messages": [llm.invoke(state["messages"])]}

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "What is a directed graph, in one sentence?"}]})
print(result["messages"][-1].content)

## ツールの追加: ノード、条件分岐、そしてループ

モデルにツールを使わせるために、あらかじめ用意された2つの部品を追加します。`ToolNode`は、モデルが要求したツールを実行するノードです。`tools_condition`は、モデルがツールを使いたいときにはtoolsノードへ、処理が完了したときにはendへとフローを送る、既製のルーターです。

ツール自体は、意図的に2つの異なる場所から取得します。検索ツールは既製品を使います。`langchain-community`には、何百ものサービス向けにあらかじめ用意されたツールが含まれており、`GoogleSerperRun`は、すでにキーを持っている同じSerper APIをラップしています。プッシュ通知ツールは、Lab 1のときと同じように、`@tool`デコレータを使って自分で書きます。どちらの種類のツールも、まったく同じ方法でグラフに組み込まれることに注目してください。

ちなみに、インポートを実行したときに非推奨(deprecation)の警告が出たことに気づいたかもしれません。`langchain-community`パッケージは、`langchain-openai`が単独で提供されているのと同じように、統合ごとに独立したパッケージへと徐々に移行しつつあります。それでも今のところは問題なく動作し、ウェブ上のあちこちのチュートリアルでも目にするので、知っておく価値はあります。完全にサポートされた代替を使いたい場合は、`langchain-tavily`パッケージが提供する`TavilySearch`ツールが、tavily.comの無料APIキーで同じ役割を果たしてくれます。

ツールをモデルに紐付けて、その存在を知らせたうえで、ループを組み立てます。chatbotが判断し、toolsが実行され、そしてchatbotに戻ってその結果を見て処理を続けられるようにします。

In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())

@tool
def send_push_notification(text: str) -> str:
    """Send a short push notification to the user's phone."""
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={"token": os.getenv("PUSHOVER_TOKEN"), "user": os.getenv("PUSHOVER_USER"), "message": text},
    )
    return "Notification sent"

tools = [search, send_push_notification]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
def chatbot_node(state: State) -> dict:
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

builder = StateGraph(State)

builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition)
builder.add_edge("tools", "chatbot")

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "Use your search tool to tell me the ingredients in banoffee pie and send me a push notification."}]})
print(result["messages"][-1].content)

## 2つ目のLLMノード: 翻訳者

ノードはtool loopの一部である必要はありません。どんな関数でもノードになれます。自分でLLM呼び出しを行うようなものでも構いません。それを証明するために、最終的な答えのスペイン語版をstateに書き込む翻訳者を追加してみましょう。

ここで新しいことが2つあります。まず、stateに2つ目のフィールド`spanish`が加わります。これにはreducerがないので、各実行でただ上書きされるだけになり、これはまさに望んでいる動作です。次に、これまで`tools_condition`は完了した会話をそのまま`END`に送っていました。第3引数としてマッピングを渡すことで、そのブランチを代わりに翻訳者に通し、翻訳者が実行を終了させます。

今回はレンダリングされたグラフをよく見てください。LLMを呼び出すノードが2つあります。1つはツールを持ち、ループの中に位置するchatbotで、もう1つは出口の途中で1回だけ普通の呼び出しを行うtranslatorです。そして、translatorはchatbotが処理を終えたあとに実行されるので、それ専用の1つのスーパーステップを持つことになります。

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    spanish: str

def translator_node(state: State) -> dict:
    last = state["messages"][-1].content
    prompt = f"Translate this into Spanish, replying with the translation only:\n\n{last}"
    return {"spanish": llm.invoke(prompt).content}

builder = StateGraph(State)

builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)

builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END:"translator"})
builder.add_edge("tools", "chatbot")
builder.add_edge("translator", END)
graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
result = graph.invoke({"messages": [{"role": "user", "content": "In one sentence, what is special about a banana?"}]})
print(result["messages"][-1].content)
print(result["spanish"])

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">LangSmithによる可観測性（Observability）</h2>
            <span style="color:#00bfff;">LangSmithはすべてのモデル呼び出しとツール呼び出しを記録するので、自分のグラフが正確に何をしたのかを確認できます。<a href="https://smith.langchain.com">smith.langchain.com</a>にサインアップしてAPIキーを作成し、以下の行を<code>.env</code>ファイルに追加してください。<br/><br/>
            <code>LANGSMITH_TRACING=true</code><br/>
            <code>LANGSMITH_ENDPOINT=https://api.smith.langchain.com</code></br>
            <code>LANGSMITH_API_KEY=lsv2_...</code><br/>
            <code>LANGSMITH_PROJECT=agentic-track</code><br/><br/>
            それだけです。これで、LangChainとLangGraphのすべての実行が、コードを変更することなく自動的にトレースされるようになり、LangSmithのダッシュボードでその展開の様子を見ることができます。ここでは実際には実行しませんが、自分の作業では有効にしておく価値が十分にあります。
            </span>
        </td>
    </tr>
</table>

In [ ]:
# LangSmithの変数を追加した場合は、ここでもう一度envを読み込みます

load_dotenv(override=True)

## メモリ: なぜグラフは忘れてしまうのか、そしてその直し方

上のグラフと2回チャットしてみると、最初のやり取りを覚えていないことが分かります。stateは丁寧にメッセージを受け渡していたはずなので、これは少し奇妙に感じられます。その理由は、`invoke`の1回の呼び出しがグラフの1回の実行に対応しているからです。実行とは、実行される各ノードの層ごとに1つずつ対応する、一連のスーパーステップです（LLM呼び出しは1つのスーパーステップであり、それが要求するツール呼び出しは次のスーパーステップでまとめて実行されます）。reducerはその実行の中でメッセージを積み重ねますが、次の`invoke`は新たにやり直します。

呼び出しをまたいで記憶させるには、checkpointerを追加します。これは、各スーパーステップの後にstateのスナップショットを保存し、`thread_id`のもとに整理します。次回も同じ`thread_id`を渡せば、新しいメッセージが追加される前に、保存された履歴が読み込まれます。

In [ ]:
memory = MemorySaver()

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)
builder.add_edge(START, "chatbot")
builder.add_conditional_edges("chatbot", tools_condition, {"tools": "tools", END: "translator"})
builder.add_edge("translator", END)
builder.add_edge("tools", "chatbot")
graph = builder.compile(checkpointer=memory)


config = {"configurable": {"thread_id": "conversation-1"}}
graph.invoke({"messages": [{"role": "user", "content": "Hi, my name is Ed."}]}, config)
second = graph.invoke({"messages": [{"role": "user", "content": "What is my name?"}]}, config)
print(second["messages"][-1].content)
print(second["spanish"])

## SQLiteへの永続化

`MemorySaver`はすべてをメモリ上に保持しますが、これはプログラムが停止すると消えてしまいます。何か永続的なものが欲しい場合は、データベースファイルに書き込むSQLite checkpointerに切り替えます。同じグラフのコードが、これで再起動しても残るようになります。このノートブックの隣に`memory.db`というファイルが現れますが、いつでも削除して構いません。

In [ ]:
with SqliteSaver.from_conn_string("memory.db") as sql_memory:
    durable_graph = builder.compile(checkpointer=sql_memory)
    sql_config = {"configurable": {"thread_id": "conversation-2"}}
    durable_graph.invoke({"messages": [{"role": "user", "content": "Remember that my favorite color is orange."}]}, sql_config)
    reply = durable_graph.invoke({"messages": [{"role": "user", "content": "What is my favorite color?"}]}, sql_config)
    print(reply["messages"][-1].content)
    print(reply["spanish"])

## stateの内部を見る、そして時間をさかのぼる

checkpointerは各スーパーステップでスナップショットを保存するので、現在のstateを調べたり、履歴全体をたどったりできます。各エントリには`checkpoint_id`が付いており、そのうちの1つをconfigに入れることで、その正確な時点からグラフを再実行できます。これによって、LangGraphアプリケーションは、以前のどの時点からでも復旧したり分岐したりできるのです。

In [ ]:
snapshot = graph.get_state(config)
messages = snapshot.values["messages"]
print("Messages stored so far:", len(messages))

for message in messages:
    print(message.content)

history = list(graph.get_state_history(config))
print("Number of saved checkpoints:", len(history))


In [ ]:
for h in reversed(history):            # historyは新しい順なので、reversedで物語のように読める
      m = h.metadata
      queued = ", ".join(t.name for t in h.tasks) or "(run complete)"
      print(f"step {m['step']:>2}  {m['source']:<5}  messages={len(h.values.get('messages', []))}  about to run: {queued}")

In [ ]:
# タイムトラベル: 以前のcheckpointを取り出し、その正確な時点からグラフを再開する
earlier = history[len(history) // 2]
print("A checkpoint from earlier held", len(earlier.values["messages"]), "messages")

replay_config = {"configurable": {"thread_id": "conversation-1",
                                  "checkpoint_id": earlier.config["configurable"]["checkpoint_id"]}}
resumed = graph.invoke(None, replay_config)
print("Resumed from the past; the graph now holds", len(resumed["messages"]), "messages")

again = graph.invoke({"messages": [{"role": "user", "content": "What do you know about me?"}]}, replay_config)
print(again["messages"][-1].content)
print(again["spanish"])


## UIの追加

シンプルなGradio UIを追加して、これに命を吹き込みましょう。`chat`関数は、メッセージごとにグラフの実行を1回開始し、固定の`thread_id`を使うことで会話が記憶されるようにし、各返答の下にイタリック体でスペイン語訳を表示します。

In [ ]:
def chat(user_input: str, history):
    config = {"configurable": {"thread_id": "gradio-session4"}}
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config)
    return f"{result['messages'][-1].content}\n\n*{result['spanish']}*"

gr.ChatInterface(chat).launch()

## まとめ、そしてこれから向かう先

ここまでで、reducerを備えたstate、ノード、エッジ、条件分岐エッジ、tool loop、2つ目のLLMノード、可観測性(observability)、2種類のメモリ、そして全体を調べたり再現したりする方法を、自分の手でグラフとして組み立ててきました。これが、その上に築かれるすべてを動かす仕組みです。

明日はレイヤー3に到達します。制御から便利さへと向かう道筋の、次のステップです。`create_agent`に、上で作ったtool loopとまったく同じグラフを、たった1行で作らせてみます。そして、そのグラフを描画し、それが自分たちが作ったものと同じであることを確認します。

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">演習</h2>
            <span style="color:#ff7800;">グラフに3つ目のツールを追加し、検索のあとにプッシュ通知を必要とし、さらに自分のツールも使うような質問をアシスタントに投げかけて、tool loopが複数回実行されるようにしてみましょう。そして、有効にしていればLangSmithダッシュボードを開き、どのノードがどの順序で実行されたかを正確にトレースしてみてください。
            </span>
        </td>
    </tr>
</table>